# Hardware y Escala — Módulo 15

Este notebook es el compañero práctico del módulo de arquitectura. Aquí vas a:

1. Inspeccionar el hardware de tu propia máquina desde Python
2. Ver la diferencia de vectorización con tus propios ojos (y temporizador)
3. Observar efectos de caché en código real
4. Medir FLOPs empíricamente en multiplicaciones de matrices
5. Reproducir las gráficas del módulo y explorar los datos
6. Estimar costos de entrenamiento de LLMs interactivamente

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/15_arquitectura_de_computadoras/code/01_arquitectura.ipynb)

In [15]:
%pip install numpy matplotlib psutil -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import platform
import psutil
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # para compatibilidad en Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print(f"Python:    {platform.python_version()}")
print(f"NumPy:     {np.__version__}")
print(f"Plataforma: {platform.system()} {platform.machine()}")

Python:    3.12.4
NumPy:     2.4.4
Plataforma: Linux x86_64


---
## Sección 1: Tu hardware

Python puede interrogar al sistema operativo sobre el hardware disponible.
Esto es útil para entender en qué entorno corre tu código y qué puedes esperar de él.

In [ ]:
# Información de CPU
print("=== CPU ===")
# 'platform.machine()' devuelve la arquitectura del sistema,
# es decir, el "idioma" que habla tu procesador: x86_64 (64 bits),
# i386 (32 bits), arm64 (ARM de 64 bits), etc.
# Esto define cómo se ejecutan los programas y qué tipo de instrucciones entiende la CPU.
print(f"Arquitectura:       {platform.machine()}")
print(f"Procesador:         {platform.processor() or 'N/D (ver /proc/cpuinfo)'}") #nombre modelo del procesador
print(f"Núcleos físicos:    {psutil.cpu_count(logical=False)}") #numero de prcoesadores fisicos
print(f"Núcleos lógicos:    {psutil.cpu_count(logical=True)}")
freq = psutil.cpu_freq()
if freq:
    print(f"Frecuencia actual:  {freq.current:.0f} MHz")
    print(f"Frecuencia máx:     {freq.max:.0f} MHz") # AMD Precision Boost ajusta la frecuencia dinámicamente,
# por lo que el sistema no expone una frecuencia máxima fija (aparece como 0)

=== CPU ===
Arquitectura:       x86_64
Procesador:         x86_64
Núcleos físicos:    12
Núcleos lógicos:    24
Frecuencia actual:  1996 MHz
Frecuencia máx:     0 MHz


In [ ]:
# Información de memoria y almacenamiento
print("=== Memoria ===")
ram = psutil.virtual_memory() #esto devuelve los valores en bytes
print(f"RAM total:          {ram.total / 2**30:.1f} GB") #divides para pasarlo a GB
print(f"RAM disponible:     {ram.available / 2**30:.1f} GB") #el :.1f solo formatea para ver un decimal
print(f"RAM en uso:         {ram.percent:.1f}%")

print("\n=== Almacenamiento ===")
# Itera sobre las primeras 2 particiones del sistema
for part in psutil.disk_partitions()[:2]:
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"{part.mountpoint}: {usage.total / 2**30:.0f} GB total, "
              f"{usage.free / 2**30:.0f} GB libres")
    # Ignora particiones sin permisos de lectura
    except PermissionError:
        pass

=== Memoria ===
RAM total:          15.2 GB
RAM disponible:     12.5 GB
RAM en uso:         17.8%

=== Almacenamiento ===
/: 1007 GB total, 942 GB libres
/mnt/wslg/distro: 1007 GB total, 942 GB libres


In [6]:
# ¿Hay GPU disponible?
print("=== GPU ===")
try:
    import torch
    if torch.cuda.is_available():
        print(f"CUDA disponible: SÍ")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory
        print(f"VRAM: {mem / 2**30:.1f} GB")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("Metal (Apple GPU): disponible")
    else:
        print("Sin GPU acelerada disponible — usando CPU")
except ImportError:
    print("PyTorch no instalado — no se puede verificar GPU")
    print("En Colab: Runtime > Change runtime type > GPU para activarla")

=== GPU ===
Sin GPU acelerada disponible — usando CPU


Como dice, instalé PyTorch pero investigue y dice que dado que el entorno es WSL2 con una GPU AMD, no es posible acceder 
a la aceleración por hardware, por lo que se utilizará CPU.

In [5]:
!pip install torch torchvision torchaudio

  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-ma

> **💡 Prueba esto:** Compara tus resultados con los de alguien más en la clase.
> ¿Cuántos núcleos físicos vs lógicos tiene cada máquina? ¿Cuánta RAM?
> En Colab (CPU runtime): ¿cuántos núcleos obtienes? ¿Cuánto RAM?

Mi computadora cuenta con 12 núcleos físicos y 24 núcleos lógicos al igual que cuenta con 15.2 GB de RAM (aunque solo esten disponibles 12.9 GB). Mi computadora es un Asus Zenbook S16.
Por otra parte, 

---
## Sección 2: Vectorización — ver la diferencia

Esta es la demostración más importante del notebook.
Vamos a hacer exactamente la misma operación de tres formas y medir el tiempo.

In [20]:
N = 10_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

N = 10,000,000
Listas Python: 188888890 objetos
Arrays NumPy:  80 MB + 80 MB


#### Diferencia entre definir las listas o arreglos con numpy vs con listas

**Lista Python `list(range(N))`**

Cada número es un **objeto independiente** en memoria, y la lista guarda *punteros* (direcciones) hacia cada uno:

```
lista → [ptr] → objeto int (28 bytes)
        [ptr] → objeto int (28 bytes)
        [ptr] → objeto int (28 bytes)
        ...
```

- Cada entero Python ocupa ~28 bytes aunque sea el número `1`
- Los objetos están **dispersos** por la memoria
- 10 millones de números → ~280 MB

---

**Array NumPy `np.arange(N)`**

Los números se guardan **uno tras otro** en un bloque continuo de memoria, sin objetos intermedios:

```
array → [8 bytes][8 bytes][8 bytes][8 bytes]...
```

- Cada número ocupa exactamente 8 bytes (float64)
- Todo está **junto** en memoria
- 10 millones de números → 80 MB

---

**¿Por qué importa esto más allá de la memoria?**

Cuando haces una operación como `a + b`:

- **Lista Python** → Python hace un loop interno, va objeto por objeto, convierte cada uno
- **NumPy** → manda la operación directo al procesador en **C/Fortran**, opera todo el bloque de una vez

Por eso NumPy puede ser **10x a 100x más rápido** para cálculos matemáticos con muchos datos. Eso es exactamente lo que el código está a punto de medir.

Se crean dos conjuntos de datos de 10 millones de elementos cada uno,
uno como listas nativas de Python y otro como arrays de NumPy,
para comparar su rendimiento en memoria y velocidad de cómputo.

La diferencia principal es que las listas de Python almacenan cada número
como un objeto independiente (mayor uso de memoria), mientras que los arrays
de NumPy almacenan los datos en bloques contiguos de memoria con tipo fijo,
lo que los hace más eficientes y rápidos para operaciones numéricas.

In [21]:
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")

Loop Python:    1.469 s


Se mide el tiempo de sumar dos listas de 10 millones de elementos
usando un loop puro de Python. Este método es el más lento ya que
Python procesa cada elemento uno por uno, tardando 1.112 segundos. No está optimizado, busca cada elemento, lo suma, etc.

In [9]:
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"

NumPy:          0.1011 s


Se realiza la misma suma pero usando NumPy vectorizado. En lugar de procesar
elemento por elemento, NumPy opera sobre todo el array de una sola vez
en bloques contiguos de memoria usando código optimizado en C por debajo.

Por eso tardó solo 0.2324 segundos vs 1.112 segundos del loop Python —
casi 5 veces más rápido. El `assert` al final verifica que ambos métodos
den el mismo resultado.

In [10]:
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")


=== Resumen ===
Loop Python:   1.191 s
NumPy:         0.1011 s
Speedup:       12×

NumPy es 12 veces más rápido con los mismos datos y la misma operación.
La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.


Se calcula el speedup dividiendo el tiempo del loop entre el tiempo de NumPy.
El resultado muestra que NumPy fue 5 veces más rápido, gracias a tres factores:

- **SIMD**: el procesador ejecuta varias operaciones simultáneamente
- **Sin overhead de objetos Python**: NumPy trabaja directamente con números en memoria
- **Memoria contigua**: los datos están juntos en memoria, lo que acelera el acceso

> **💡 Prueba esto:**
> - Cambia `N` a 100_000 y a 100_000_000. ¿Cómo cambia el speedup?
> - Prueba con `np.float32` en vez de `np.float64`. ¿Es más rápido?
> - ¿Qué pasa si haces `a_np * b_np` (multiplicación) en vez de suma?

#### Vamos a hacer la prueba con 100_000_000

In [12]:
N = 100_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

N = 100,000,000
Listas Python: 2088888890 objetos
Arrays NumPy:  800 MB + 800 MB


Se tardó 42.9 segundos solo en generar mientras que con 10_000_000 nse había tardado 1.7s.

In [13]:
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")

Loop Python:    17.985 s


Se aumentó N a 100 millones de elementos para observar cómo escala
el rendimiento. El loop Python tardó 17.985 segundos, confirmando
que su tiempo crece linealmente con el tamaño de los datos.

In [14]:
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"

NumPy:          1.3636 s


Con 100 millones de elementos NumPy tardó solo 1.3636 segundos frente
a los 17.985 segundos del loop Python, resultando en un speedup de ~13x.
Notablemente, la ventaja de NumPy aumenta con el tamaño de los datos:
con 10 millones era 5x más rápido, con 100 millones es 13x,
lo que demuestra que NumPy escala mejor que Python puro.

In [15]:
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")


=== Resumen ===
Loop Python:   17.985 s
NumPy:         1.3636 s
Speedup:       13×

NumPy es 13 veces más rápido con los mismos datos y la misma operación.
La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.


Con 100 millones de elementos, el speedup final fue de 13x.
NumPy opera sobre los datos en bloque usando código optimizado en C,
mientras que el loop Python procesa cada elemento individualmente,
acumulando overhead en cada iteración.

#### Vamos a hacer la prueba con 100_000

In [16]:
N = 100_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

N = 100,000
Listas Python: 1488890 objetos
Arrays NumPy:  1 MB + 1 MB


Ahora se tardó 6.9 segundos, es más de lo que tardó haciendo 1_000_000. Probablemente porque todos los datos anteriores están usando la RAM.

In [17]:
import psutil
ram = psutil.virtual_memory()
print(f"RAM disponible: {ram.available / 2**30:.1f} GB")

RAM disponible: 8.9 GB


Aquí notamos que la RAM sí decreció después de hacer el ejercicio anterior. Por eso vamos a limpiar la RAM.

In [18]:
import psutil, gc

print(f"Antes: {psutil.virtual_memory().available / 2**30:.1f} GB disponibles")

del a, b, a_np, b_np, result_loop, result_np
gc.collect()

print(f"Después: {psutil.virtual_memory().available / 2**30:.1f} GB disponibles")

Antes: 8.9 GB disponibles
Después: 13.3 GB disponibles


Vuelvo a generar la celda de los 100_000.

In [19]:
N = 100_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

N = 100,000
Listas Python: 1488890 objetos
Arrays NumPy:  1 MB + 1 MB


Ahora se tardó prácticamente 0 segundos. Se nota la mejoría vs los casi 7 segundos de la otra ocasión sin haber limpiado la RAM.

In [20]:
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")


Loop Python:    0.009 s
NumPy:          0.0094 s

=== Resumen ===
Loop Python:   0.009 s
NumPy:         0.0094 s
Speedup:       1×

NumPy es 1 veces más rápido con los mismos datos y la misma operación.
La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.


Con 100,000 elementos el speedup fue de 1x, es decir, NumPy y el loop Python
tardaron prácticamente lo mismo. Esto se debe a que con datasets pequeños
el overhead de inicializar NumPy cancela su ventaja. NumPy es más eficiente
a partir de millones de elementos, donde la vectorización y la memoria
contigua marcan una diferencia significativa.

#### Vamos a probar con np.float32

Vamos a limpiar la RAM primero.

In [11]:
import psutil, gc

print(f"Antes: {psutil.virtual_memory().available / 2**30:.1f} GB disponibles")

del a, b, a_np, b_np, result_loop, result_np
gc.collect()

print(f"Después: {psutil.virtual_memory().available / 2**30:.1f} GB disponibles")

Antes: 11.1 GB disponibles
Después: 12.4 GB disponibles


Vamos ahora a volver a los 10_000_000. Pero con np.float32.

In [ ]:
N = 10_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float32)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float32)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")

N = 10,000,000
Listas Python: 188888890 objetos
Arrays NumPy:  40 MB + 40 MB
Loop Python:    0.847 s
NumPy:          0.0325 s

=== Resumen ===
Loop Python:   0.847 s
NumPy:         0.0325 s
Speedup:       26×

NumPy es 26 veces más rápido con los mismos datos y la misma operación.
La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.


Al cambiar a float32 (4 bytes por elemento vs 8 de float64), los arrays
ocuparon solo 40 MB cada uno en lugar de 80 MB. Esto permitió al procesador
meter más datos en caché y aprovechar mejor las instrucciones SIMD,
aumentando el speedup de 5x a 26x con los mismos 10 millones de elementos.

Float32 y float64 son tipos de datos para números decimales. float64 usa
64 bits (8 bytes) y ofrece hasta 15 decimales de precisión, mientras que
float32 usa 32 bits (4 bytes) y ofrece hasta 7 decimales. Al usar float32
se reduce a la mitad el uso de memoria, lo que acelera las operaciones
ya que el procesador puede manejar más datos simultáneamente.

El speedup aumentó de 5x a 26x al cambiar a float32 porque NumPy aprovecha
las instrucciones SIMD del procesador, que operan sobre varios datos a la vez.
Con float64 el procesador procesa 4 números por operación, mientras que con
float32 procesa 8, ya que cada número ocupa la mitad del espacio. Esto significa
que NumPy hace el mismo trabajo en la mitad de operaciones. El loop Python
no se beneficia de esto porque no usa SIMD.

NumPy utiliza instrucciones SIMD del procesador, que operan sobre múltiples
datos a la vez usando registros de memoria fija (256 bits en AVX2).
Al dividir ese espacio entre el tamaño de cada número se obtiene cuántos
elementos se procesan por operación:

- float64 (64 bits): 256 / 64 = 4 números por operación
- float32 (32 bits): 256 / 32 = 8 números por operación

Al reducir el tamaño del número a la mitad, NumPy procesa el doble de
elementos por operación. El AMD Ryzen AI 9 puede tener AVX-512 (512 bits),
lo que duplicaría estos valores y explicaría el alto speedup observado.

#### Ahora vamos a hacer`a_np * b_np` (multiplicación) en vez de suma

Limpiamos RAM:

In [18]:
import psutil, gc

print(f"Antes: {psutil.virtual_memory().available / 2**30:.1f} GB disponibles")

del a, b, a_np, b_np, result_loop, result_np
gc.collect()

print(f"Después: {psutil.virtual_memory().available / 2**30:.1f} GB disponibles")

Antes: 11.5 GB disponibles


NameError: name 'result_loop' is not defined

Ahora si hacemos la multiplicacion:

In [29]:
N = 10_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] * b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np * b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")

N = 10,000,000
Listas Python: 188888890 objetos
Arrays NumPy:  80 MB + 80 MB
Loop Python:    0.914 s
NumPy:          0.0623 s

=== Resumen ===
Loop Python:   0.914 s
NumPy:         0.0623 s
Speedup:       15×

NumPy es 15 veces más rápido con los mismos datos y la misma operación.
La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.


Se realizó la multiplicación con ambos tipos de datos:

- float32: speedup de 162x (NumPy tardó 0.0069 s vs 1.117 s del loop)
- float64: speedup de 15x (NumPy tardó 0.0623 s vs 0.914 s del loop)

La diferencia es mayor en la multiplicación que en la suma porque
la multiplicación es una operación más costosa para el loop Python,
ya que requiere más ciclos de procesador por objeto. NumPy la optimiza
igual de bien con SIMD, ampliando la brecha respecto al loop Python.

Además, float32 ocupa 4 bytes vs 8 bytes de float64, lo que permite
a SIMD procesar el doble de elementos por operación, explicando por qué
el speedup con float32 es mucho mayor que con float64.

---
## Sección 3: Efectos de caché

Acceder a datos en orden secuencial (cache-friendly) es dramáticamente más rápido
que acceder de forma aleatoria, aunque el número de operaciones sea idéntico.

Esto no es sobre Python — es sobre física del hardware.

In [ ]:
# Creamos una matriz cuadrada grande
SIZE = 2000
M = np.random.randn(SIZE, SIZE).astype(np.float32) #genera números aleatorios siguiendo una distribución normal estándar
# (SIZE, SIZE) define el tamaño de la matriz y astype solo define el tipo de números que van a ser
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")

Matriz: 2000×2000 = 4,000,000 elementos
Tamaño en memoria: 16.0 MB


In [27]:
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM 
    #practicamente dice fila 1, todas las columnas, aqui defines que se recorre por fila
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")

Acceso por filas:    0.0128 s  (total=-629.01)


## ¿Cómo guarda NumPy los datos en memoria?

Cuando creamos una matriz en NumPy, los datos no se guardan como objetos
separados dispersos por la RAM, sino en un **bloque continuo de memoria**,
es decir, cajita tras cajita sin saltos ni huecos.

Por ejemplo, esta matriz:
```python
M = [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]
```
Se guarda así en RAM:
```
Dirección:  100  101  102  103  104  105  106  107  108
Valor:      [ 1 ][ 2 ][ 3 ][ 4 ][ 5 ][ 6 ][ 7 ][ 8 ][ 9 ]
             <-- fila 0 --> <-- fila 1 --> <-- fila 2 -->
```

Esto se llama **row-major order**: primero todos los elementos de la fila 0,
luego la fila 1, luego la fila 2, y así sucesivamente.

### ¿Por qué importa esto?

Cuando el procesador pide un dato a la RAM, no trae solo ese dato — trae
un bloque de posiciones vecinas al **caché** (memoria ultra-rápida del CPU).
Si los datos están contiguos, los siguientes elementos **ya están en caché**
cuando los necesitas. Esto se llama acceso **cache-friendly**.

Por eso recorrer una matriz **fila por fila** es rápido: cada fila es un
bloque continuo y el procesador la carga de un jalón.

Recorrerla **columna por columna** es más lento: los elementos de una columna
están separados en memoria, entonces cada acceso provoca un **cache miss**
(el procesador tiene que ir a buscar a la RAM en lugar de usar el caché).

### Comparación con listas Python

Una lista Python guarda **punteros** que apuntan a objetos dispersos por
toda la RAM, en posiciones no predecibles. NumPy en cambio siempre mantiene
sus datos juntos, lo que lo hace significativamente más eficiente para
operaciones matemáticas sobre grandes volúmenes de datos.

In [28]:
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].sum()  # leer columna completa — saltos en memoria
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

Acceso por columnas: 0.0221 s  (total=-629.01)

Ratio columnas/filas: 1.73×
(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)


### ¿Por qué ocurre esto?

Cuando NumPy lee `M[i, :]` (una fila), los datos están contiguos en memoria —
el procesador los carga en caché en bloques de 64 bytes (cache lines).
Las siguientes lecturas ya están en caché.

Cuando lee `M[:, j]` (una columna), cada elemento está a `SIZE × 4 = 8,000 bytes`
del anterior. Cada acceso es un cache miss: hay que ir a buscar a RAM.

```
Fila [i, :]:   [0][1][2][3][4]...  ← contiguos, 1 cache miss por bloque
Columna [:, j]: [0]....[1]....[2].. ← separados, 1 cache miss por elemento
```

En pandas: `.iterrows()` accede por filas a un DataFrame que internamente
almacena columnas. Por eso es especialmente lento.

> **💡 Prueba esto:**
> - Cambia `SIZE` a 200. ¿Desaparece el efecto? (La matriz cabe en caché L3)
> - Prueba con `SIZE = 5000`. ¿Se amplifica el efecto?
> - ¿Qué pasa si conviertes la columna a array contiguo antes: `M[:, j].copy().sum()`?

## Vamos a cambiar el size a 200

In [29]:
# Creamos una matriz cuadrada grande
SIZE = 200
M = np.random.randn(SIZE, SIZE).astype(np.float32) #genera números aleatorios siguiendo una distribución normal estándar
# (SIZE, SIZE) define el tamaño de la matriz y astype solo define el tipo de números que van a ser
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM 
    #practicamente dice fila 1, todas las columnas, aqui defines que se recorre por fila
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].sum()  # leer columna completa — saltos en memoria
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

Matriz: 200×200 = 40,000 elementos
Tamaño en memoria: 0.2 MB
Acceso por filas:    0.0006 s  (total=19.22)
Acceso por columnas: 0.0007 s  (total=19.22)

Ratio columnas/filas: 1.16×
(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)


El efecto no desaparece pero el ratio es menor, supongo que porque son menos elementos. Pasó de ser un ratio de 1.73, es decir que tomaba cerca del doble de tiempo mientras que con menos elementos es de 1.16.

## Ahora vamos con SIZE = 5000

In [31]:
# Creamos una matriz cuadrada grande
SIZE = 5000
M = np.random.randn(SIZE, SIZE).astype(np.float32) #genera números aleatorios siguiendo una distribución normal estándar
# (SIZE, SIZE) define el tamaño de la matriz y astype solo define el tipo de números que van a ser
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM 
    #practicamente dice fila 1, todas las columnas, aqui defines que se recorre por fila
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].sum()  # leer columna completa — saltos en memoria
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

Matriz: 5000×5000 = 25,000,000 elementos
Tamaño en memoria: 100.0 MB
Acceso por filas:    0.0272 s  (total=2658.43)
Acceso por columnas: 0.0778 s  (total=2658.42)

Ratio columnas/filas: 2.86×
(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)


Conforme aumenta el número de elementos la diferencia se hace más marcada, supongo que porque para cada elemento debe de buscar en espacios disjuntos de la memoria, por lo que se tarda más. Ahora se volvió un ratio de 2.86 a comparación del primero que era de 1.73.

## Ahora transformamos la columna a array contiguo

In [33]:
# Creamos una matriz cuadrada grande
SIZE = 2000
M = np.random.randn(SIZE, SIZE).astype(np.float32) #genera números aleatorios siguiendo una distribución normal estándar
# (SIZE, SIZE) define el tamaño de la matriz y astype solo define el tipo de números que van a ser
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM 
    #practicamente dice fila 1, todas las columnas, aqui defines que se recorre por fila
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].copy().sum()  # leer columna, la transformamos en array contiguo
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

Matriz: 2000×2000 = 4,000,000 elementos
Tamaño en memoria: 16.0 MB
Acceso por filas:    0.0127 s  (total=195.55)
Acceso por columnas: 0.0196 s  (total=195.55)

Ratio columnas/filas: 1.54×
(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)


Copy es que recorre la columna dispersa y después la agrupa. AL final para sumarlos no va a salir tan caro en cuestion de trabajo y de tiempo porque los datos ya estarán juntos pero de todas maneras se tiene que recorrer la memoria una vez para poder agruparlos. Esto sigue teniendo un costo pero aparentemente el costo es menor que si lo hicieras discontiguo como antes.

---
## Sección 4: Multiplicación de matrices y FLOPs

La operación central de las redes neuronales. Vamos a medirla empíricamente
y calcular cuántos FLOPs realmente estamos ejecutando.

In [41]:
def benchmark_matmul(N, reps=3):
    """Mide tiempo de N×N matmul y calcula GFLOPS observados."""
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)

    # Warm-up
    _ = A @ B #elimina la primera multiplicacion porque esta es más lenta generalmente por razones del sistema

    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        C = A @ B
        times.append(time.perf_counter() - t0)

    t_med = np.median(times)
    flops = 2 * N**3           # FLOPs teóricos: 2×M×N×K con M=N=K
    gflops = flops / t_med / 1e9
    return t_med, gflops

sizes = [64, 128, 256, 512, 1024, 2048]
print(f"{'N':>6}  {'Tiempo (ms)':>12}  {'GFLOPS obs.':>12}")
print("-" * 36)
results = []
for N in sizes:
    t, g = benchmark_matmul(N)
    results.append((N, t, g))
    print(f"{N:>6}  {t*1000:>12.2f}  {g:>12.1f}")

     N   Tiempo (ms)   GFLOPS obs.
------------------------------------
    64          0.02          27.0
   128          0.09          48.3
   256          2.83          11.8
   512          3.37          79.5
  1024         10.03         214.1
  2048         43.96         390.8


## Análisis de resultados — Multiplicación de matrices

Para matrices pequeñas (N=64, 128, 256) los GFLOPS son bajos e inconsistentes
a pesar de tener un warm-up previo. Esto se debe a que el cálculo es tan
rápido (~0.03ms) que las **interrupciones del sistema operativo** — que pueden
durar ~1ms — dominan la medición. Para obtener resultados confiables en estos
tamaños se necesitarían muchas más repeticiones (100+) para que la mediana
absorba ese ruido.

A partir de N=512 los resultados son confiables y se estabilizan entre
**300 y 400 GFLOPS**, rango donde el cálculo tarda lo suficiente para que
las interrupciones del OS no distorsionen la medición.

Para N=2048 se alcanzó el mejor resultado con **401 GFLOPS**, porque las
matrices son suficientemente grandes para que NumPy active optimizaciones
internas más agresivas, dividiendo el trabajo en bloques que aprovechan
el caché de forma más eficiente.

En conclusión, el CPU muestra un rendimiento sostenido de ~300-400 GFLOPS
para multiplicación de matrices en float32. Para comparación, una GPU
entry-level supera los 5,000 GFLOPS, lo que explica por qué el entrenamiento
de modelos de deep learning se realiza en GPU.

In [38]:
# Graficar GFLOPS observados vs N
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), facecolor="#1a1a2e")

for ax in [ax1, ax2]:
    ax.set_facecolor("#16213e")
    for sp in ax.spines.values():
        sp.set_edgecolor("#2a2a4e")
    ax.tick_params(colors="#e0e0e0")
    ax.xaxis.label.set_color("#e0e0e0")
    ax.yaxis.label.set_color("#e0e0e0")
    ax.title.set_color("#e0e0e0")
    ax.yaxis.grid(True, color="#2a2a4e", alpha=0.5)
    ax.set_axisbelow(True)

ns     = [r[0] for r in results]
times  = [r[1] * 1000 for r in results]  # ms
gflops = [r[2] for r in results]

ax1.plot(ns, times, 'o-', color="#0db7ed", linewidth=2, markersize=7)
ax1.set_xlabel("Tamaño de matriz N")
ax1.set_ylabel("Tiempo (ms)")
ax1.set_title("Tiempo de N×N matmul")

ax2.plot(ns, gflops, 's-', color="#f0a500", linewidth=2, markersize=7)
ax2.set_xlabel("Tamaño de matriz N")
ax2.set_ylabel("GFLOPS observados")
ax2.set_title("GFLOPS observados — crece con N")

fig.suptitle("Multiplicación de matrices: tiempo y GFLOPS",
             color="#e0e0e0", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("matmul_benchmark.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()
print("\n¿Por qué los GFLOPS crecen con N?")
print("Con matrices más grandes, la GPU (o BLAS) puede saturar sus núcleos.")
print("Matrices pequeñas tienen overhead de setup relativo mayor.")


¿Por qué los GFLOPS crecen con N?
Con matrices más grandes, la GPU (o BLAS) puede saturar sus núcleos.
Matrices pequeñas tienen overhead de setup relativo mayor.


/tmp/ipykernel_8659/1733488152.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **💡 Prueba esto:**
> - Compara `np.float32` vs `np.float64`. ¿Cuánto cambia el rendimiento?
> - Si tienes GPU disponible: prueba con `torch.matmul` en CUDA y compara.
> - ¿Qué GFLOPS teóricos tiene tu CPU? (Busca el modelo y sus specs.)

## Hacemos el analisis en float 64

In [44]:
def benchmark_matmul(N, reps=3):
    """Mide tiempo de N×N matmul y calcula GFLOPS observados."""
    A = np.random.randn(N, N).astype(np.float64)
    B = np.random.randn(N, N).astype(np.float64)

    # Warm-up
    _ = A @ B #elimina la primera multiplicacion porque esta es más lenta generalmente por razones del sistema

    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        C = A @ B
        times.append(time.perf_counter() - t0)

    t_med = np.median(times)
    flops = 2 * N**3           # FLOPs teóricos: 2×M×N×K con M=N=K
    gflops = flops / t_med / 1e9
    return t_med, gflops

sizes = [64, 128, 256, 512, 1024, 2048]
print(f"{'N':>6}  {'Tiempo (ms)':>12}  {'GFLOPS obs.':>12}")
print("-" * 36)
results = []
for N in sizes:
    t, g = benchmark_matmul(N)
    results.append((N, t, g))
    print(f"{N:>6}  {t*1000:>12.2f}  {g:>12.1f}")

     N   Tiempo (ms)   GFLOPS obs.
------------------------------------
    64          0.02          28.9
   128          0.12          34.8
   256          3.69           9.1
   512          7.74          34.7
  1024         22.58          95.1
  2048         93.62         183.5


## Comparación float32 vs float64 — Multiplicación de matrices

| N    | GFLOPS float32 | GFLOPS float64 | Ratio |
|------|---------------|----------------|-------|
| 64   | 20.6          | 28.9           | 0.7×  |
| 128  | 13.4          | 34.8           | 0.4×  |
| 256  | 38.6          | 9.1            | 4.2×  |
| 512  | 322.2         | 34.7           | 9.3×  |
| 1024 | 301.1         | 95.1           | 3.2×  |
| 2048 | 401.7         | 183.5          | 2.2×  |

Para matrices pequeñas (N=64, 128) los resultados siguen siendo poco confiables
en ambos casos por el mismo motivo anterior: el tiempo de cálculo es tan corto
que el ruido del sistema operativo domina la medición.

A partir de N=512 la diferencia es clara y consistente: **float32 es
aproximadamente 2× a 9× más rápido que float64**. Esto se explica por dos
razones. Primero, float32 ocupa 4 bytes por número contra 8 bytes de float64,
por lo que caben el doble de elementos en caché simultáneamente. Segundo, el
procesador puede operar el doble de números por ciclo usando instrucciones
SIMD, ya que el "carril" de procesamiento es fijo en bits.

El caso más llamativo es N=512 donde float32 alcanza 322 GFLOPS contra apenas
34.7 GFLOPS de float64 (ratio 9.3×), muy por encima del 2× teórico esperado.
Esto sugiere que en ese tamaño la matriz float32 cabía completamente en caché
mientras que float64 ya no, amplificando la diferencia.

En conclusión, para cálculos donde la precisión de ~7 decimales es suficiente
(como redes neuronales o gráficas), float32 es claramente superior. Para
cálculos científicos o financieros que requieren ~15 decimales de precisión,
float64 es necesario asumiendo el costo de rendimiento.

## GFLOPs Teoricos

El Zenbook S16 viene en varias configuraciones de CPU. El modelo más común es el **AMD Ryzen AI 9 HX 370**, así que asumo que tienes ese (si tienes otro dime cuál y te recalculo).

Aquí están los **GFLOPS teóricos** del Ryzen AI 9 HX 370:

---

### 🔢 CPU — Rendimiento teórico pico

Calculado con 4 núcleos de rendimiento a 5.1 GHz, AVX2 de 256 bits con doble FMA:

| Precisión | GFLOPS |
|---|---|
| **FP64** (doble) | ~326 GFLOPS |
| **FP32** (simple) | ~653 GFLOPS |
| **BF16** | ~1,310 GFLOPS (1.31 TFLOPS) |
| **INT8** | ~2,610 TOPS (2.61 TOPS) |

---

### 🎮 iGPU (Radeon 890M) — lo más relevante para cómputo

La GPU integrada suele superar al CPU en GFLOPS. El Radeon 890M tiene 16 CUs RDNA 3.5 corriendo hasta ~2,900 MHz, lo que da aproximadamente **~3.0 TFLOPS FP32** teóricos.

---

### 🤖 NPU

El sistema completo (CPU + GPU + NPU) ofrece 80 TOPS, con la NPU aportando 50 TOPS por sí sola.

---

**Resumen rápido:** para cómputo de propósito general el CPU da ~653 GFLOPS FP32, pero la iGPU (Radeon 890M) es mucho más potente con ~3 TFLOPS FP32. 

---
## Sección 5: Reproduce el gráfico histórico de FLOPs

Los datos del módulo, reproducibles y modificables.

In [5]:
# Datos históricos: (año, nombre, USD por GFLOP)
# Fuentes: diversos estudios de Karl Rupp, Our World in Data, benchmarks públicos
flops_history = [
    (1961, "IBM 7090",           1e10),
    (1984, "Cray X-MP",          4.2e7),
    (1994, "Intel Pentium",      3e4),
    (1997, "Intel Pentium II",   1e3),
    (2001, "AMD Athlon XP",      1e2),
    (2006, "PlayStation 3",      1.0),
    (2008, "AMD Radeon HD 4870", 0.065),
    (2013, "NVIDIA GTX 780",     0.002),
    (2017, "NVIDIA GTX 1080",    3.5e-4),
    (2020, "NVIDIA RTX 3090",    4e-5),
    (2022, "NVIDIA RTX 4090",    1.5e-5),
    (2023, "NVIDIA H100",        2e-6),
]

years = [d[0] for d in flops_history]
costs = [d[2] for d in flops_history]
names = [d[1] for d in flops_history]

reduction = flops_history[0][2] / flops_history[-1][2]
print(f"Reducción total (1961 → 2023): {reduction:,.0f}×")

Reducción total (1961 → 2023): 5,000,000,000,000,000×


In [6]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.set_axisbelow(True)

# Colorear: CPU = azul, GPU = violeta, consola = verde
colors = [
    "#0db7ed", "#0db7ed", "#0db7ed", "#0db7ed", "#0db7ed",  # CPUs
    "#2ecc71",                                                # PS3
    "#892ca0", "#892ca0", "#892ca0", "#892ca0", "#892ca0", "#892ca0",  # GPUs
]

ax.plot(years, costs, color="#444466", linewidth=1, alpha=0.5, zorder=1)
for i, (yr, cost, name) in enumerate(zip(years, costs, names)):
    ax.scatter(yr, cost, color=colors[i], s=80, zorder=3,
               edgecolors="white", linewidths=0.5)
    ax.annotate(name, (yr, cost),
                xytext=(5, 3), textcoords="offset points",
                color="#e0e0e0", fontsize=8)

ax.set_yscale("log")
ax.set_xlabel("Año")
ax.set_ylabel("USD por GFLOP (escala log)")
ax.set_title(f"Costo histórico de 1 GFLOP — {reduction:,.0f}× más barato en 62 años",
             fontweight="bold")

import matplotlib.patches as mpatches
leg = [
    mpatches.Patch(color="#0db7ed", label="CPU"),
    mpatches.Patch(color="#892ca0", label="GPU"),
    mpatches.Patch(color="#2ecc71", label="Consola"),
]
ax.legend(handles=leg, facecolor="#16213e", labelcolor="#e0e0e0", edgecolor="#2a2a4e")

plt.tight_layout()
plt.savefig("flops_historico_notebook.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

/tmp/ipykernel_6126/3708879331.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **💡 Prueba esto:**
> - Agrega chips nuevos que encuentres online (MI300X, RTX 5090, etc.)
> - Calcula el precio del chip actual y divídelo entre los TFLOPS para obtener USD/GFLOP
> - ¿Qué costo por GFLOP tiene tu propia computadora?

## Ejercicio de agregar chips nuevos y calcular el precio del chip actual y dividirlo entre los TFLOPS

In [8]:
# Datos históricos + chips nuevos (2024-2025)
# USD por GFLOP (FP32), precio del chip / GFLOPS totales
flops_history = [
    (1961, "IBM 7090",              1e10),
    (1984, "Cray X-MP",             4.2e7),
    (1994, "Intel Pentium",         3e4),
    (1997, "Intel Pentium II",      1e3),
    (2001, "AMD Athlon XP",         1e2),
    (2006, "PlayStation 3",         1.0),
    (2008, "AMD Radeon HD 4870",    0.065),
    (2013, "NVIDIA GTX 780",        0.002),
    (2017, "NVIDIA GTX 1080",       3.5e-4),
    (2020, "NVIDIA RTX 3090",       4e-5),
    (2022, "NVIDIA RTX 4090",       1.5e-5),
    (2023, "NVIDIA H100",           2e-6),

    # ── Chips nuevos ──────────────────────────────────────────
    # AMD MI300X: ~$10,000 retail enterprise / 163,400 GFLOPS FP32
    (2023, "AMD MI300X",            10_000 / 163_400),   # ~$0.0000612

    # NVIDIA RTX 5090: $1,999 MSRP / 104,800 GFLOPS FP32
    (2025, "NVIDIA RTX 5090",       1_999 / 104_800),    # ~$0.0000191

    # ── Tu computadora ────────────────────────────────────────
    # Radeon 890M (iGPU del Ryzen AI 9 HX 370)
    # 16 CUs × 64 shaders × 2 ops × 2900 MHz = ~5,939 GFLOPS FP32 (~5.9 TFLOPS)
    # El Zenbook S16 cuesta ~$1,399 USD (precio base)
    # Pero la iGPU no tiene precio separado — se divide el costo del laptop
    # entre los GFLOPS de la iGPU para tener un estimado comparable
    (2024, "Radeon 890M (Zenbook S16)", 1_399 / 5_939),  # ~$0.000236
]

years = [d[0] for d in flops_history]
costs = [d[2] for d in flops_history]
names = [d[1] for d in flops_history]

reduction = flops_history[0][2] / flops_history[-1][2]
print(f"Reducción total (1961 → 2025): {reduction:,.0f}×")

# Imprimir tabla completa
print(f"\n{'Año':<6} {'Nombre':<30} {'USD/GFLOP':>15}")
print("-" * 55)
for año, nombre, costo in flops_history:
    print(f"{año:<6} {nombre:<30} ${costo:>14.6f}")

# La reducción depende del chip final que elijas:
h100_cost = 2e-6
rtx5090_cost = 1_999 / 104_800  # ~1.91e-5

print(f"H100 (datacenter):  ${h100_cost:.2e}/GFLOP")
print(f"RTX 5090 (consumer): ${rtx5090_cost:.2e}/GFLOP")
print(f"La RTX 5090 es {rtx5090_cost/h100_cost:.0f}× más cara por GFLOP que la H100")
# → La RTX 5090 es ~10× más cara por GFLOP que la H100

Reducción total (1961 → 2025): 42,451,751,251×

Año    Nombre                               USD/GFLOP
-------------------------------------------------------
1961   IBM 7090                       $10000000000.000000
1984   Cray X-MP                      $42000000.000000
1994   Intel Pentium                  $  30000.000000
1997   Intel Pentium II               $   1000.000000
2001   AMD Athlon XP                  $    100.000000
2006   PlayStation 3                  $      1.000000
2008   AMD Radeon HD 4870             $      0.065000
2013   NVIDIA GTX 780                 $      0.002000
2017   NVIDIA GTX 1080                $      0.000350
2020   NVIDIA RTX 3090                $      0.000040
2022   NVIDIA RTX 4090                $      0.000015
2023   NVIDIA H100                    $      0.000002
2023   AMD MI300X                     $      0.061200
2025   NVIDIA RTX 5090                $      0.019074
2024   Radeon 890M (Zenbook S16)      $      0.235562
H100 (datacenter):  $2.00e-

La reducción cambia dependiendo del chip que agarremos porque el ultimo que escogimos es mas caro que el H100, por lo tanto la reducción es menor.

---
## Sección 6: Explorador de costos de LLMs

Una función para estimar cuánto costaría entrenar un modelo dado.

In [9]:
def estimar_entrenamiento(
    params_B: float,          # parámetros del modelo en billones
    tokens_B: float,          # tokens de entrenamiento en billones
    gpu_tflops_bf16: float,   # rendimiento efectivo del GPU en TFLOPS BF16
    gpu_price_per_hour: float = 3.0,  # USD por hora por GPU
    gpu_efficiency: float = 0.35,     # eficiencia real (35% típico en clúster)
) -> dict:
    """
    Estima FLOPs, tiempo y costo de entrenamiento usando la fórmula de Chinchilla:
    FLOPs ≈ 6 × parámetros × tokens
    """
    params = params_B * 1e9
    tokens = tokens_B * 1e9

    flops_total = 6 * params * tokens

    # FLOPS/s efectivos de 1 GPU
    flops_per_sec_gpu = gpu_tflops_bf16 * 1e12 * gpu_efficiency

    tiempo_segundos = flops_total / flops_per_sec_gpu
    tiempo_horas = tiempo_segundos / 3600
    costo_usd = tiempo_horas * gpu_price_per_hour

    return {
        "flops_total": flops_total,
        "flops_total_str": f"{flops_total:.2e}",
        "gpu_horas": tiempo_horas,
        "costo_1_gpu_usd": costo_usd,
        "gpus_para_1_semana": tiempo_horas / (24 * 7),
    }

print("Función lista. Probemos con algunos ejemplos...")

Función lista. Probemos con algunos ejemplos...


In [10]:
# H100 con ~1,000 TFLOPS BF16 (rendimiento pico)
H100_TFLOPS = 1000
H100_PRICE  = 3.0   # USD/hora (cloud spot)

ejemplos = [
    ("LLaMA 2 7B",   7,    1_400),   # 7B params, ~1.4T tokens
    ("LLaMA 2 70B",  70,   2_000),   # 70B params, 2T tokens
    ("GPT-3 175B",   175,  300),     # 175B params, 300B tokens
    ("Modelo 1B (tuyo)", 1, 20),     # 1B params, 20B tokens — un proyecto pequeño
]

print(f"{'Modelo':<22} {'FLOPs':>10}  {'GPU-horas':>12}  {'Costo 1 GPU':>14}  {'GPUs p/semana':>14}")
print("-" * 78)

for nombre, params_B, tokens_B in ejemplos:
    r = estimar_entrenamiento(params_B, tokens_B, H100_TFLOPS, H100_PRICE)
    print(f"{nombre:<22} {r['flops_total_str']:>10}  "
          f"{r['gpu_horas']:>12,.0f}  "
          f"${r['costo_1_gpu_usd']:>12,.0f}  "
          f"{r['gpus_para_1_semana']:>14,.0f}")

Modelo                      FLOPs     GPU-horas     Costo 1 GPU   GPUs p/semana
------------------------------------------------------------------------------
LLaMA 2 7B               5.88e+22        46,667  $     140,000             278
LLaMA 2 70B              8.40e+23       666,667  $   2,000,000           3,968
GPT-3 175B               3.15e+23       250,000  $     750,000           1,488
Modelo 1B (tuyo)         1.20e+20            95  $         286               1


In [11]:
# Visualizar: parámetros vs costo estimado
modelos_viz = [
    {"name": "BERT-Large",   "params_B": 0.34,  "flops": 1.5e20, "cost_M": 0.007},
    {"name": "GPT-2",        "params_B": 1.5,   "flops": 5e19,   "cost_M": 0.05},
    {"name": "T5-11B",       "params_B": 11,    "flops": 5e21,   "cost_M": 0.5},
    {"name": "GPT-3 175B",   "params_B": 175,   "flops": 3.1e23, "cost_M": 5},
    {"name": "PaLM 540B",    "params_B": 540,   "flops": 2.5e24, "cost_M": 50},
    {"name": "LLaMA 2 70B",  "params_B": 70,    "flops": 2e24,   "cost_M": 20},
    {"name": "GPT-4 (est.)", "params_B": 1800,  "flops": 2e25,   "cost_M": 100},
]

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.xaxis.grid(True, color="#2a2a4e", alpha=0.2)
ax.set_axisbelow(True)

colors_m = plt.cm.plasma(np.linspace(0.1, 0.9, len(modelos_viz)))
for i, m in enumerate(modelos_viz):
    ax.scatter(m["params_B"], m["cost_M"], s=100, color=colors_m[i],
               zorder=3, edgecolors="white", linewidths=0.5)
    ax.annotate(m["name"], (m["params_B"], m["cost_M"]),
                xytext=(6, 4), textcoords="offset points",
                color="#e0e0e0", fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Parámetros (billones)")
ax.set_ylabel("Costo de entrenamiento (millones USD)")
ax.set_title("Parámetros vs Costo de entrenamiento", fontweight="bold")

plt.tight_layout()
plt.savefig("llm_costo_notebook.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

/tmp/ipykernel_6126/2484734864.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **💡 Prueba esto:**
> - Llama a `estimar_entrenamiento` con tu propio modelo hipotético.
> - ¿Cuántas GPUs H100 necesitarías para terminar en 1 semana un modelo de 13B con 1T tokens?
> - ¿Qué pasa si cambias la eficiencia de 35% a 50%? ¿Qué hace que no sea 100%?

## Propio modelo

In [ ]:
mi_modelo = estimar_entrenamiento(
    params_B=10,        # 10 miles de millones
    tokens_B=1_500,     # 1500 miles de millones = 1.5 billones de tokens
    gpu_tflops_bf16=989,
    gpu_price_per_hour=3.0,
    gpu_efficiency=0.35
)

for k, v in mi_modelo.items():
    print(f"{k}: {v}")

flops_total: 9e+22
flops_total_str: 9.00e+22
gpu_horas: 72223.02470027444
costo_1_gpu_usd: 216669.07410082332
gpus_para_1_semana: 429.89895654925266


## ¿Cuantas GPUs H100 necesitarías para terminar en 1 semana un modelo de 13 B con 1T tokens?

In [14]:
nuevo_modelo = estimar_entrenamiento(
    params_B=13,        
    tokens_B=1_000, 
    gpu_tflops_bf16=989,
    gpu_price_per_hour=3.0,
    gpu_efficiency=0.35
)

for k, v in nuevo_modelo.items():
    print(f"{k}: {v}")

flops_total: 7.8e+22
flops_total_str: 7.80e+22
gpu_horas: 62593.288073571195
costo_1_gpu_usd: 187779.86422071359
gpus_para_1_semana: 372.579095676019


## Eficiencia del 50%

In [15]:
nuevo_modelo = estimar_entrenamiento(
    params_B=13,        
    tokens_B=1_000, 
    gpu_tflops_bf16=989,
    gpu_price_per_hour=3.0,
    gpu_efficiency=0.5
)

for k, v in nuevo_modelo.items():
    print(f"{k}: {v}")

flops_total: 7.8e+22
flops_total_str: 7.80e+22
gpu_horas: 43815.30165149983
costo_1_gpu_usd: 131445.9049544995
gpus_para_1_semana: 260.80536697321327


Con mejor eficiencia necesitas menos GPUs para poder realizar todo el trabajo en una semana. Por el cuello de botella en el traslado de la memoria a la GPU no es posible eficiencia del 100%, la H100 puede hacer 989 TFLOPS BF16, pero su VRAM entrega datos a 3.35 TB/s. El problema es que esas dos velocidades no están balanceadas — la GPU calcula más rápido de lo que puede recibir datos. Entonces las unidades de cómputo terminan su operación y se quedan esperando los siguientes pesos o activaciones. Es como una impresora industrial que puede imprimir 1000 páginas por minuto pero el papel llega en rollos que se desenrollan a 300 páginas por minuto — la máquina espera aunque físicamente sea rapidísima.

---
## Ejercicio integrador

Tienes acceso a un clúster con 8 GPUs H100 (1,000 TFLOPS BF16 cada una, rendimiento
efectivo 35%, $3/hora cada una). Quieres entrenar un modelo de lenguaje.

In [13]:
# Tu código aquí

# 1. Calcula cuántos parámetros puedes entrenar en máximo 1 semana con 8 GPUs
#    si usas el ratio óptimo de Chinchilla (20 tokens por parámetro).
def parametros_max(
    gpus: int,              # número de GPUs disponibles
    gpu_tflops_bf16: float,   # rendimiento efectivo del GPU en TFLOPS BF16
    gpu_price_per_hour: float = 3.0,  # USD por hora por GPU
    gpu_efficiency: float = 0.35,     # eficiencia real (35% típico en clúster)
    costo_max_usd: float = None           # presupuesto máximo en USD (opcional)
) -> dict:
    """
    Estima el número máximo de parámetros que se pueden entrenar en una semana con 8 GPUs,
    aplicando la fórmula de Chinchilla y considerando el precio y eficiencia de las GPUs.
    """
    # Usamos el ratio óptimo de Chinchilla (20 tokens por parámetro)
    # Sabemos que los FLOPs totales deben ser menores iguales a lo que 8 GPUs pueden procesar en una semana
    # flops_totales = 6 × params × tokens
    # A su vez, sabemos que tokens = 20 × params (Chinchilla), entonces:
    # flops_totales = 6 × params × (20 × params) = 120 × params^2
    # Tambien, el tiempo disponible en una semana con 8 GPUs es:
    # tiempo_disponible = 8 GPUs × 7 días × 24 horas × 3600 segundos
    # Y los flops sobre segundo son:     flops_per_sec_gpu = gpu_tflops_bf16 * 1e12 * gpu_efficiency
    # Entonces calculamos el tiempo disponible en segundos y lo calculamos por los flops por segundo para tener el total de flops

    flops_per_sec_gpu = gpu_tflops_bf16 * 1e12 * gpu_efficiency
    tiempo_disponible_segundos = 3600 * 24 * 7 * gpus
    parametros_maximos = (tiempo_disponible_segundos * flops_per_sec_gpu / 120) ** 0.5
    costo = tiempo_disponible_segundos * (gpu_price_per_hour / 3600)
    tiempo_disponible_maximo = costo_max_usd / (gpu_price_per_hour * gpus) if costo_max_usd is not None else tiempo_disponible_segundos / 3600
    parametros_con_presupuesto_max = (tiempo_disponible_maximo * 3600 * gpus * flops_per_sec_gpu / 120) ** 0.5 if costo_max_usd is not None else parametros_maximos

    return {
    "flops_por_segundo_gpu": flops_per_sec_gpu,
    "tiempo_disponible_segundos": tiempo_disponible_segundos,
    "parametros_maximos": parametros_maximos,
    "parametros_maximos_B": parametros_maximos / 1e9,  # en miles de millones
    "tokens_optimos_B": (parametros_maximos * 20) / 1e9,  # tokens de Chinchilla
    "costo": f"${costo:,.2f} USD",
    "tiempo_disponible_maximo_horas": tiempo_disponible_maximo,
    "tamaño_modelo_con_presupuesto_max_B": parametros_con_presupuesto_max / 1e9,

    }

#Resolvemos para 8 GPUs H100 con 1000 TFLOPS BF16 efectivos
resultado = parametros_max(
    gpus=8,
    gpu_tflops_bf16=1000,
    gpu_price_per_hour=3.0,
    gpu_efficiency=0.35,
    costo_max_usd=10_000  # presupuesto máximo de $10,000 USD
)
for k, v in resultado.items():
    print(f"{k}: {v}")

# 2. ¿Cuánto cuesta ese entrenamiento en total?
# $ 4032 USD

# 3. Si el presupuesto máximo es $10,000 USD, ¿qué tamaño de modelo puedes entrenar?
#    (sigue asumiendo Chinchilla y 8 GPUs H100)
# Aqui lo que cambia es el tiempo que puedo rentar las GPUs entonces el tiempo disponible se calcula con el dinero.

# 4. Benchmarkea en tu máquina: ¿cuántos GFLOPS obtienes en matmul 1024×1024?
#    ¿Cuánto tiempo tardaría en tu CPU hacer lo que un H100 hace en 1 segundo?
import numpy as np
import time

N = 1024
ops = 2 * N**3  # FLOPs de una multiplicación de matrices

A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

t0 = time.perf_counter()
C = np.dot(A, B)
t = time.perf_counter() - t0

gflops = ops / t / 1e9
print(f"Mi CPU: {gflops:.1f} GFLOPS")

h100_gflops = 989_000  # 989 TFLOPS = 989,000 GFLOPS
mi_cpu_gflops = gflops  # el número que mediste arriba

ratio = h100_gflops / mi_cpu_gflops
print(f"Lo que H100 hace en 1 segundo, mi CPU lo hace en {ratio:.0f} segundos")
print(f"O sea {ratio/60:.1f} minutos, o {ratio/3600:.2f} horas")

# Pista para el punto 1:
# flops_totales = 6 × params × tokens
# tokens = 20 × params  (Chinchilla)
# tiempo_disponible = 8 GPUs × 7 días × 24 horas × 3600 segundos × TFLOPS × eficiencia
# despeja params de: flops_totales ≤ tiempo_disponible × FLOPS/s

flops_por_segundo_gpu: 350000000000000.0
tiempo_disponible_segundos: 4838400
parametros_maximos: 3756594202.1996465
parametros_maximos_B: 3.7565942021996466
tokens_optimos_B: 75.13188404399294
costo: $4,032.00 USD
tiempo_disponible_maximo_horas: 416.6666666666667
tamaño_modelo_con_presupuesto_max_B: 5.916079783099616
Mi CPU: 27.9 GFLOPS
Lo que H100 hace en 1 segundo, mi CPU lo hace en 35392 segundos
O sea 589.9 minutos, o 9.83 horas


<details>
<summary>Pista para el punto 1</summary>

```python
n_gpus = 8
h100_tflops = 1000
eficiencia = 0.35
semanas = 1

flops_disponibles = n_gpus * h100_tflops * 1e12 * eficiencia * semanas * 7 * 24 * 3600

# FLOPs = 6 × params × 20 × params = 120 × params²
# params = sqrt(FLOPs / 120)
import math
params = math.sqrt(flops_disponibles / 120)
print(f"Parámetros máximos: {params/1e9:.1f}B")
```
</details>

---
## Resumen

| Experimento | Lección |
|-------------|----------|
| Hardware inspection | Cada máquina tiene restricciones concretas: cores, RAM, ausencia de GPU |
| Vectorización | NumPy no es solo comodidad — usa instrucciones SIMD del hardware |
| Efectos de caché | El orden de acceso a memoria importa tanto como el número de operaciones |
| MatMul FLOPs | Matrices grandes saturan el hardware; pequeñas son overhead-bound |
| FLOPs histórico | El costo de cómputo cayó 5 billones de veces en 62 años |
| Escala LLMs | GPT-4 costó ~$100M en compute. LLaMA 2 7B: estimado en ~$3M. Fine-tuning: miles. |

**El mensaje central:** el hardware no es un detalle de implementación.
Es el presupuesto físico dentro del cual vive cada algoritmo.